In [ ]:
import cv2
import json
import numpy as np
import os
from roboflow import Roboflow
# --- CONFIGURACIÓN ---
API_KEY = "tuapikey"  # <--- PEGA TU API KEY
IMAGE_PATH = "tuimg"
OUTPUT_IMAGE = "resultado_puntos_final.jpg" # Nombre de la imagen que guardaremos
# --- 1. FUNCIÓN DE CARGA ROBUSTA ---
def cargar_imagen_robusta(ruta):
    if not os.path.exists(ruta): return None
    if ruta.lower().endswith('.insp'):
        try:
            with open(ruta, 'rb') as f: bytes_img = bytearray(f.read())
            numpy_array = np.asarray(bytes_img, dtype=np.uint8)
            return cv2.imdecode(numpy_array, cv2.IMREAD_COLOR)
        except: return None
    return cv2.imread(ruta)
# --- 2. INICIALIZAR MODELO ---
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("agridrone-pblcc").project("agridetect")
model = project.version(3).model
# --- 3. PROCESAMIENTO ---
print(f"Cargando imagen {IMAGE_PATH}...")
image_numpy = cargar_imagen_robusta(IMAGE_PATH)
if image_numpy is not None:
    height, width, _ = image_numpy.shape
    # Preparamos imagen para inferencia (RGB)
    image_rgb = cv2.cvtColor(image_numpy, cv2.COLOR_BGR2RGB)
    # Inferencia con confianza baja (5%) para intentar captar señales débiles
    print("Ejecutando IA...")
    response = model.predict(image_rgb, confidence=5).json()
    # Normalizar lista
    if isinstance(response, dict) and 'predictions' in response:
        detections = response['predictions']
    elif isinstance(response, list):
        detections = response
    else:
        detections = []
    # --- 4. LÓGICA HÍBRIDA (IA + GEOMETRÍA) ---
    # A. CULTIVO (Prioridad: IA -> Fallback: Abajo Centro)
    soil_labels = ["field-soil", "unused-land", "agriculture-land", "trees", "crop", "soil"]
    best_soil = None
    max_conf_soil = 0
    for det in detections:
        if det['class'] in soil_labels:
            if det['confidence'] > max_conf_soil:
                max_conf_soil = det['confidence']
                best_soil = det
    soil_result = None
    if best_soil:
        soil_result = {
            "x": int(best_soil['x']),
            "y": int(best_soil['y'])
        }
    else:
        soil_result = {
            "x": int(width / 2),
            "y": int(height * 0.75), # 75% hacia abajo
        }
    # B. CIELO (Prioridad: IA -> Fallback: Arriba Centro)
    sky_labels = ["sky", "cielo", "cloud"]
    best_sky = None
    max_conf_sky = 0
    for det in detections:
        if det['class'] in sky_labels:
            if det['confidence'] > max_conf_sky:
                max_conf_sky = det['confidence']
                best_sky = det
    sky_result = None
    if best_sky:
        sky_result = {
            "x": int(best_sky['x']),
            "y": int(best_sky['y'])
        }
    else:
        sky_result = {
            "x": int(width / 2),
            "y": int(height * 0.15), # 15% hacia abajo (Cenit)
        }
    # --- 5. DIBUJAR RESULTADOS EN LA IMAGEN ---
    print("Dibujando marcadores...")
    # Hacemos una copia para pintar
    img_vis = image_numpy.copy()
    def dibujar_marcador(img, data, color, label_principal):
        x, y = data["x"], data["y"]
        # 1. Dibujar círculo sólido grande
        cv2.circle(img, (x, y), 60, color, -1)
        # 2. Dibujar borde blanco al círculo para contraste
        cv2.circle(img, (x, y), 60, (255, 255, 255), 5)
        # 3. Poner Texto con borde negro (para que se lea en cualquier fondo)
        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 3.0
        thickness = 8
        text_size, _ = cv2.getTextSize(texto, font, scale, thickness)
        text_x = x - text_size[0] // 2 # Centrar texto
        text_y = y - 80 # Poner texto un poco arriba del punto
        cv2.putText(img, texto, (text_x, text_y), font, scale, (0, 0, 0), thickness + 4) # Borde negro
        cv2.putText(img, texto, (text_x, text_y), font, scale, (255, 255, 255), thickness) # Texto blanco
    # DIBUJAR CIELO (AZUL - BGR: 255, 0, 0)
    if sky_result:
        dibujar_marcador(img_vis, sky_result, (255, 0, 0), "CIELO")
    # DIBUJAR SUELO (VERDE - BGR: 0, 255, 0)
    if soil_result:
        dibujar_marcador(img_vis, soil_result, (0, 255, 0), "CULTIVO")
    # --- 6. GUARDAR IMAGEN Y SALIDA JSON ---
    cv2.imwrite(OUTPUT_IMAGE, img_vis)
    final_json = {
        "cielo": sky_result,
        "cultivo": soil_result
    }
    print("\n--- JSON GENERADO ---")
    print(json.dumps(final_json, indent=4))
    print(f"\n--- IMAGEN GUARDADA ---")
    print(f"Revisa el archivo: {OUTPUT_IMAGE}")
else:
    print(json.dumps({"error": "No se pudo cargar la imagen"}))

RuntimeError: {
    "error": {
        "message": "This API key does not exist (or has been revoked).",
        "status": 401,
        "type": "OAuthException",
        "hint": "You may retrieve your API key via the Roboflow Dashboard. Go to Account > Roboflow Keys to retrieve yours.",
        "key": "tuapikey"
    }
}

In [ ]:
import cv2
import json
import numpy as np
import os
from roboflow import Roboflow
from dotenv import load_dotenv

# --- 0. CARGAR VARIABLES DE ENTORNO ---
load_dotenv()  # Carga el archivo .env

API_KEY = os.getenv("ROBOFLOW_API_KEY")
if not API_KEY:
    raise ValueError("❌ No se encontró ROBOFLOW_API_KEY en el archivo .env")

# --- CONFIGURACIÓN ---
OUTPUT_IMAGE = "resultado_puntos_final.jpg"  # Nombre de la imagen de salida

# Pedir imagen por input
IMAGE_PATH = input("Introduce la ruta de la imagen: ").strip()

# --- 1. FUNCIÓN DE CARGA ROBUSTA ---
def cargar_imagen_robusta(ruta):
    if not os.path.exists(ruta):
        print("❌ La ruta no existe:", ruta)
        return None

    if ruta.lower().endswith('.insp'):
        try:
            with open(ruta, 'rb') as f:
                bytes_img = bytearray(f.read())
            numpy_array = np.asarray(bytes_img, dtype=np.uint8)
            return cv2.imdecode(numpy_array, cv2.IMREAD_COLOR)
        except Exception as e:
            print("❌ Error leyendo archivo .insp:", e)
            return None

    img = cv2.imread(ruta)
    if img is None:
        print("❌ OpenCV no pudo leer la imagen.")
    return img

# --- 2. INICIALIZAR MODELO ---
print("Inicializando Roboflow...")
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("agridrone-pblcc").project("agridetect")
model = project.version(3).model

# --- 3. PROCESAMIENTO ---
print(f"\nCargando imagen: {IMAGE_PATH}")
image_numpy = cargar_imagen_robusta(IMAGE_PATH)

if image_numpy is not None:
    height, width, _ = image_numpy.shape

    # Convertir a RGB para inferencia
    image_rgb = cv2.cvtColor(image_numpy, cv2.COLOR_BGR2RGB)

    # Inferencia con confianza baja (5%)
    print("Ejecutando IA...")
    response = model.predict(image_rgb, confidence=5).json()

    # Normalizar lista de detecciones
    if isinstance(response, dict) and 'predictions' in response:
        detections = response['predictions']
    elif isinstance(response, list):
        detections = response
    else:
        detections = []

    # --- 4. LÓGICA HÍBRIDA (IA + GEOMETRÍA) ---

    # A. CULTIVO (Prioridad: IA -> Fallback: Abajo Centro)
    soil_labels = ["field-soil", "unused-land", "agriculture-land", "trees", "crop", "soil"]
    best_soil = None
    max_conf_soil = 0

    for det in detections:
        if det['class'] in soil_labels:
            if det['confidence'] > max_conf_soil:
                max_conf_soil = det['confidence']
                best_soil = det

    if best_soil:
        soil_result = {
            "x": int(best_soil['x']),
            "y": int(best_soil['y'])
        }
    else:
        soil_result = {
            "x": int(width / 2),
            "y": int(height * 0.75),  # 75% hacia abajo
            }

    # B. CIELO (Prioridad: IA -> Fallback: Arriba Centro)
    sky_labels = ["sky", "cielo", "cloud"]
    best_sky = None
    max_conf_sky = 0

    for det in detections:
        if det['class'] in sky_labels:
            if det['confidence'] > max_conf_sky:
                max_conf_sky = det['confidence']
                best_sky = det

    if best_sky:
        sky_result = {
            "x": int(best_sky['x']),
            "y": int(best_sky['y']),
        }
    else:
        sky_result = {
            "x": int(width / 2),
            "y": int(height * 0.15),  # 15% hacia abajo
        }

    # --- 5. DIBUJAR RESULTADOS EN LA IMAGEN ---
    print("Dibujando marcadores...")

    img_vis = image_numpy.copy()

    def dibujar_marcador(img, data, color, label_principal):
        x, y = data["x"], data["y"]

        # 1. Círculo sólido
        cv2.circle(img, (x, y), 60, color, -1)

        # 2. Borde blanco
        cv2.circle(img, (x, y), 60, (255, 255, 255), 5)

        # 3. Texto con borde
        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 3.0
        thickness = 8

        text_size, _ = cv2.getTextSize(texto, font, scale, thickness)
        text_x = x - text_size[0] // 2


Inicializando Roboflow...
loading Roboflow workspace...
loading Roboflow project...

Cargando imagen: prueba.insp
Ejecutando IA...
Dibujando marcadores...


In [ ]:
import cv2
import json
import numpy as np
import os
from roboflow import Roboflow
from dotenv import load_dotenv

# --- 0. CARGAR VARIABLES DE ENTORNO ---
load_dotenv()

API_KEY = os.getenv("ROBOFLOW_API_KEY")
if not API_KEY:
    raise ValueError("❌ No se encontró ROBOFLOW_API_KEY en el archivo .env")

# --- CONFIGURACIÓN ---
OUTPUT_IMAGE = "resultado_puntos_final.jpg"
CONVERTED_JPG = "convertida.jpg"

# Pedir imagen por input
IMAGE_PATH = input("Introduce la ruta de la imagen (.insp o .jpg): ").strip()

# --- 1. FUNCIÓN: CONVERTIR .insp A JPG ---
def convertir_insp_a_jpg(ruta_insp, ruta_salida):
    print("Intentando convertir .insp a .jpg...")

    try:
        with open(ruta_insp, 'rb') as f:
            bytes_img = bytearray(f.read())

        numpy_array = np.asarray(bytes_img, dtype=np.uint8)
        img = cv2.imdecode(numpy_array, cv2.IMREAD_COLOR)

        if img is None:
            print("❌ No se pudo decodificar el archivo .insp como imagen.")
            return None

        cv2.imwrite(ruta_salida, img)
        print(f"✅ Imagen convertida y guardada como: {ruta_salida}")
        return img

    except Exception as e:
        print("❌ Error al convertir .insp:", e)
        return None

# --- 2. FUNCIÓN: CARGA ROBUSTA ---
def cargar_imagen(ruta):
    if not os.path.exists(ruta):
        print("❌ La ruta no existe:", ruta)
        return None

    ext = os.path.splitext(ruta)[1].lower()

    if ext == ".insp":
        return convertir_insp_a_jpg(ruta, CONVERTED_JPG)
    else:
        img = cv2.imread(ruta)
        if img is None:
            print("❌ OpenCV no pudo leer la imagen.")
        return img

# --- 3. INICIALIZAR MODELO ---
print("Inicializando Roboflow...")
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("agridrone-pblcc").project("agridetect")
model = project.version(3).model

# --- 4. CARGAR IMAGEN ---
print(f"\nCargando imagen: {IMAGE_PATH}")
image_numpy = cargar_imagen(IMAGE_PATH)

if image_numpy is None:
    print(json.dumps({"error": "No se pudo cargar ni convertir la imagen"}))
    exit()

print("✅ Imagen cargada correctamente")
print("Shape:", image_numpy.shape)
print("Min pixel:", image_numpy.min(), "Max pixel:", image_numpy.max())

# Guardar debug de la imagen que se usará
cv2.imwrite("debug_entrada.jpg", image_numpy)
print("Imagen de entrada guardada como debug_entrada.jpg")

# --- 5. PREPARAR PARA INFERENCIA ---
height, width, _ = image_numpy.shape
image_rgb = cv2.cvtColor(image_numpy, cv2.COLOR_BGR2RGB)

# --- 6. INFERENCIA ---
print("\nEjecutando IA...")
response = model.predict(image_rgb, confidence=5).json()

# Normalizar detecciones
if isinstance(response, dict) and 'predictions' in response:
    detections = response['predictions']
elif isinstance(response, list):
    detections = response
else:
    detections = []

print(f"Detecciones encontradas: {len(detections)}")

# --- 7. LÓGICA HÍBRIDA (IA + GEOMETRÍA) ---

# A. CULTIVO
soil_labels = ["field-soil", "unused-land", "agriculture-land", "trees", "crop", "soil"]
best_soil = None
max_conf_soil = 0

for det in detections:
    if det['class'] in soil_labels:
        if det['confidence'] > max_conf_soil:
            max_conf_soil = det['confidence']
            best_soil = det

if best_soil:
    soil_result = {
        "x": int(best_soil['x']),
        "y": int(best_soil['y']),
    }
else:
    soil_result = {
        "x": int(width / 2),
        "y": int(height * 0.75),
    }

# B. CIELO
sky_labels = ["sky", "cielo", "cloud"]
best_sky = None
max_conf_sky = 0

for det in detections:
    if det['class'] in sky_labels:
        if det['confidence'] > max_conf_sky:
            max_conf_sky = det['confidence']
            best_sky = det

if best_sky:
    sky_result = {
        "x": int(best_sky['x']),
        "y": int(best_sky['y'])
    }
else:
    sky_result = {
        "x": int(width / 2),
        "y": int(height * 0.15)
    }

# --- 8. DIBUJAR RESULTADOS ---
print("Dibujando marcadores...")

img_vis = image_numpy.copy()

def dibujar_marcador(img, data, color, label_principal):
    x, y = data["x"], data["y"]

    cv2.circle(img, (x, y), 60, color, -1)
    cv2.circle(img, (x, y), 60, (255, 255, 255), 5)

    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 3.0
    thickness = 8

    text_size, _ = cv2.getTextSize(texto, font, scale, thickness)
    text_x = x - text_size[0] // 2
    text_y = y - 80

    cv2.putText(img, texto, (text_x, text_y), font, scale, (0, 0, 0), thickness + 4)
    cv2.putText(img, texto, (text_x, text_y), font, scale, (255, 255, 255), thickness)

# CIELO
dibujar_marcador(img_vis, sky_result, (255, 0, 0), "CIELO")

# CULTIVO
dibujar_marcador(img_vis, soil_result, (0, 255, 0), "CULTIVO")

# --- 9. GUARDAR RESULTADOS ---
cv2.imwrite(OUTPUT_IMAGE, img_vis)

final_json = {
    "cielo": sky_result,
    "cultivo": soil_result
}

print("\n--- JSON GENERADO ---")
print(json.dumps(final_json, indent=4))

print("\n--- IMAGEN FINAL GUARDADA ---")
print(f"Archivo: {OUTPUT_IMAGE}")


Inicializando Roboflow...
loading Roboflow workspace...
loading Roboflow project...

Cargando imagen: prueba.insp
Intentando convertir .insp a .jpg...
✅ Imagen convertida y guardada como: convertida.jpg
✅ Imagen cargada correctamente
Shape: (2944, 5888, 3)
Min pixel: 0 Max pixel: 255
Imagen de entrada guardada como debug_entrada.jpg

Ejecutando IA...
Detecciones encontradas: 13
Dibujando marcadores...

--- JSON GENERADO ---
{
    "cielo": {
        "x": 2944,
        "y": 441,
        "metodo": "FALLBACK (Geometrico)"
    },
    "cultivo": {
        "x": 3349,
        "y": 1418,
        "metodo": "IA (Detectado)"
    }
}

--- IMAGEN FINAL GUARDADA ---
Archivo: resultado_puntos_final.jpg


In [ ]:
import cv2
import json
import numpy as np
import os
from roboflow import Roboflow
from dotenv import load_dotenv

# --- 0. CARGAR VARIABLES DE ENTORNO ---
load_dotenv()

API_KEY = os.getenv("ROBOFLOW_API_KEY")
if not API_KEY:
    raise ValueError("❌ No se encontró ROBOFLOW_API_KEY en el archivo .env")

# --- CONFIGURACIÓN ---
OUTPUT_IMAGE = "resultado_puntos_final.jpg"
CONVERTED_JPG = "convertida.jpg"

# Pedir imagen por input
IMAGE_PATH = input("Introduce la ruta de la imagen (.insp o .jpg): ").strip()

# --- 1. FUNCIÓN: CONVERTIR .insp A JPG ---
def convertir_insp_a_jpg(ruta_insp, ruta_salida):
    print("Intentando convertir .insp a .jpg...")

    try:
        with open(ruta_insp, 'rb') as f:
            bytes_img = bytearray(f.read())

        numpy_array = np.asarray(bytes_img, dtype=np.uint8)
        img = cv2.imdecode(numpy_array, cv2.IMREAD_COLOR)

        if img is None:
            print("❌ No se pudo decodificar el archivo .insp como imagen.")
            return None

        cv2.imwrite(ruta_salida, img)
        print(f"✅ Imagen convertida y guardada como: {ruta_salida}")
        return img

    except Exception as e:
        print("❌ Error al convertir .insp:", e)
        return None

# --- 2. FUNCIÓN: CARGA ROBUSTA ---
def cargar_imagen(ruta):
    if not os.path.exists(ruta):
        print("❌ La ruta no existe:", ruta)
        return None

    ext = os.path.splitext(ruta)[1].lower()

    if ext == ".insp":
        return convertir_insp_a_jpg(ruta, CONVERTED_JPG)
    else:
        img = cv2.imread(ruta)
        if img is None:
            print("❌ OpenCV no pudo leer la imagen.")
        return img

# --- 3. INICIALIZAR MODELO ---
print("Inicializando Roboflow...")
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("agridrone-pblcc").project("agridetect")
model = project.version(3).model

# --- 4. CARGAR IMAGEN ---
print(f"\nCargando imagen: {IMAGE_PATH}")
image_numpy = cargar_imagen(IMAGE_PATH)

if image_numpy is None:
    print(json.dumps({"error": "No se pudo cargar ni convertir la imagen"}))
    exit()

print("✅ Imagen cargada correctamente")
print("Shape:", image_numpy.shape)
print("Min pixel:", image_numpy.min(), "Max pixel:", image_numpy.max())

# Guardar debug de la imagen que se usará
cv2.imwrite("debug_entrada.jpg", image_numpy)
print("Imagen de entrada guardada como debug_entrada.jpg")

# --- 5. PREPARAR PARA INFERENCIA ---
height, width, _ = image_numpy.shape
image_rgb = cv2.cvtColor(image_numpy, cv2.COLOR_BGR2RGB)

# --- 6. INFERENCIA ---
print("\nEjecutando IA...")
response = model.predict(image_rgb, confidence=5).json()

# Normalizar detecciones
if isinstance(response, dict) and 'predictions' in response:
    detections = response['predictions']
elif isinstance(response, list):
    detections = response
else:
    detections = []

print(f"\nDetecciones encontradas: {len(detections)}")
print("Clases detectadas por IA:")
for det in detections:
    print(f" - {det['class']} ({det['confidence']:.2f})")

# --- 7. LÓGICA HÍBRIDA MEJORADA (IA + GEOMETRÍA) ---

# A. SUELO / CULTIVO (IA + Región inferior)
soil_labels = ["field-soil", "unused-land", "agriculture-land", "trees", "crop", "soil", "plant", "vegetation"]
best_soil = None
max_conf_soil = 0

# 1. Intentar con IA primero
for det in detections:
    if det['class'] in soil_labels:
        if det['confidence'] > max_conf_soil:
            max_conf_soil = det['confidence']
            best_soil = det

if best_soil:
    soil_result = {
        "x": int(best_soil['x']),
        "y": int(best_soil['y'])
    }
else:
    # 2. Heurística: región inferior central
    print("No se detectó suelo por IA, usando heurística de región inferior...")

    y_start = int(height * 0.6)
    y_end = height
    x_start = int(width * 0.2)
    x_end = int(width * 0.8)

    x_fallback = int((x_start + x_end) / 2)
    y_fallback = int((y_start + y_end) / 2)

    soil_result = {
        "x": x_fallback,
        "y": y_fallback
    }

# B. CIELO (IA + Región superior)
sky_labels = ["sky", "cielo", "cloud"]
best_sky = None
max_conf_sky = 0

# 1. Intentar con IA
for det in detections:
    if det['class'] in sky_labels:
        if det['confidence'] > max_conf_sky:
            max_conf_sky = det['confidence']
            best_sky = det

if best_sky:
    sky_result = {
        "x": int(best_sky['x']),
        "y": int(best_sky['y'])
    }
else:
    # 2. Heurística: región superior central
    print("No se detectó cielo por IA, usando heurística de región superior...")

    y_start = 0
    y_end = int(height * 0.3)
    x_start = int(width * 0.2)
    x_end = int(width * 0.8)

    x_fallback = int((x_start + x_end) / 2)
    y_fallback = int((y_start + y_end) / 2)

    sky_result = {
        "x": x_fallback,
        "y": y_fallback
    }

# --- 8. DIBUJAR RESULTADOS ---
print("\nDibujando marcadores...")

img_vis = image_numpy.copy()

def dibujar_marcador(img, data, color, label_principal):
    x, y = data["x"], data["y"]

    # Círculo principal
    cv2.circle(img, (x, y), 60, color, -1)
    cv2.circle(img, (x, y), 60, (255, 255, 255), 5)

    # Texto
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 3.0
    thickness = 8

    text_size, _ = cv2.getTextSize(texto, font, scale, thickness)
    text_x = x - text_size[0] // 2
    text_y = y - 80

    cv2.putText(img, texto, (text_x, text_y), font, scale, (0, 0, 0), thickness + 4)
    cv2.putText(img, texto, (text_x, text_y), font, scale, (255, 255, 255), thickness)

# Dibujar cielo (azul)
dibujar_marcador(img_vis, sky_result, (255, 0, 0), "CIELO")

# Dibujar suelo (verde)
dibujar_marcador(img_vis, soil_result, (0, 255, 0), "SUELO")

# --- 9. GUARDAR RESULTADOS ---
cv2.imwrite(OUTPUT_IMAGE, img_vis)

final_json = {
    "cielo": sky_result,
    "suelo": soil_result
}

print("\n--- JSON GENERADO ---")
print(json.dumps(final_json, indent=4))

print("\n--- IMAGEN FINAL GUARDADA ---")
print(f"Archivo: {OUTPUT_IMAGE}")


Inicializando Roboflow...
loading Roboflow workspace...
loading Roboflow project...

Cargando imagen: prueba.insp
Intentando convertir .insp a .jpg...
✅ Imagen convertida y guardada como: convertida.jpg
✅ Imagen cargada correctamente
Shape: (2944, 5888, 3)
Min pixel: 0 Max pixel: 255
Imagen de entrada guardada como debug_entrada.jpg

Ejecutando IA...

Detecciones encontradas: 13
Clases detectadas por IA:
 - objects (0.49)
 - trees (0.42)
 - waterbodies (0.33)
 - trees (0.28)
 - field-soil (0.25)
 - unused-land (0.25)
 - objects (0.18)
 - waterbodies (0.13)
 - unused-land (0.09)
 - field-soil (0.09)
 - objects (0.09)
 - field-soil (0.07)
 - waterbodies (0.06)
No se detectó cielo por IA, usando heurística de región superior...

Dibujando marcadores...

--- JSON GENERADO ---
{
    "cielo": {
        "x": 2943,
        "y": 441,
        "metodo": "FALLBACK (Region Superior)"
    },
    "suelo": {
        "x": 3349,
        "y": 1418,
        "metodo": "IA (Detectado)"
    }
}

--- IMAGEN F

In [2]:
import cv2
import json
import numpy as np
import os
from roboflow import Roboflow
from dotenv import load_dotenv

# --- 0. CARGAR VARIABLES DE ENTORNO ---
load_dotenv()

API_KEY = os.getenv("ROBOFLOW_API_KEY")
if not API_KEY:
    raise ValueError("❌ No se encontró ROBOFLOW_API_KEY en el archivo .env")

# --- CONFIGURACIÓN ---
OUTPUT_IMAGE = "resultado_puntos_final.jpg"
CONVERTED_JPG = "convertida.jpg"
TREES_DIR = "arboles_detectados"

os.makedirs(TREES_DIR, exist_ok=True)

# Pedir imagen por input
IMAGE_PATH = input("Introduce la ruta de la imagen (.insp o .jpg): ").strip()

# --- 1. FUNCIÓN: CONVERTIR .insp A JPG ---
def convertir_insp_a_jpg(ruta_insp, ruta_salida):
    print("Intentando convertir .insp a .jpg...")

    try:
        with open(ruta_insp, 'rb') as f:
            bytes_img = bytearray(f.read())

        numpy_array = np.asarray(bytes_img, dtype=np.uint8)
        img = cv2.imdecode(numpy_array, cv2.IMREAD_COLOR)

        if img is None:
            print("❌ No se pudo decodificar el archivo .insp como imagen.")
            return None

        cv2.imwrite(ruta_salida, img)
        print(f"✅ Imagen convertida y guardada como: {ruta_salida}")
        return img

    except Exception as e:
        print("❌ Error al convertir .insp:", e)
        return None

# --- 2. FUNCIÓN: CARGA ROBUSTA ---
def cargar_imagen(ruta):
    if not os.path.exists(ruta):
        print("❌ La ruta no existe:", ruta)
        return None

    ext = os.path.splitext(ruta)[1].lower()

    if ext == ".insp":
        return convertir_insp_a_jpg(ruta, CONVERTED_JPG)
    else:
        img = cv2.imread(ruta)
        if img is None:
            print("❌ OpenCV no pudo leer la imagen.")
        return img

# --- 3. INICIALIZAR MODELO ---
print("Inicializando Roboflow...")
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("agridrone-pblcc").project("agridetect")
model = project.version(3).model

# --- 4. CARGAR IMAGEN ---
print(f"\nCargando imagen: {IMAGE_PATH}")
image_numpy = cargar_imagen(IMAGE_PATH)

if image_numpy is None:
    print(json.dumps({"error": "No se pudo cargar ni convertir la imagen"}))
    exit()

print("✅ Imagen cargada correctamente")
print("Shape:", image_numpy.shape)
print("Min pixel:", image_numpy.min(), "Max pixel:", image_numpy.max())

# Guardar debug de la imagen que se usará
cv2.imwrite("debug_entrada.jpg", image_numpy)
print("Imagen de entrada guardada como debug_entrada.jpg")

# --- 5. PREPARAR PARA INFERENCIA ---
height, width, _ = image_numpy.shape
image_rgb = cv2.cvtColor(image_numpy, cv2.COLOR_BGR2RGB)

# --- 6. INFERENCIA ---
print("\nEjecutando IA...")
response = model.predict(image_rgb, confidence=5).json()

# Normalizar detecciones
if isinstance(response, dict) and 'predictions' in response:
    detections = response['predictions']
elif isinstance(response, list):
    detections = response
else:
    detections = []

print(f"\nDetecciones encontradas: {len(detections)}")
print("Clases detectadas por IA:")
for det in detections:
    print(f" - {det['class']} ({det['confidence']:.2f})")

# --- 6.1 GUARDAR RECORTES DE ÁRBOLES DETECTADOS ---
tree_count = 0

for det in detections:
    if det['class'] in ["trees", "tree", "vegetation", "plant"]:
        tree_count += 1

        # Centro y tamaño del bounding box
        cx = det['x']
        cy = det['y']
        w = det['width']
        h = det['height']

        x1 = int(cx - w / 2)
        y1 = int(cy - h / 2)
        x2 = int(cx + w / 2)
        y2 = int(cy + h / 2)

        # Limitar a bordes de la imagen
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(width, x2)
        y2 = min(height, y2)

        crop = image_numpy[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        tree_filename = os.path.join(TREES_DIR, f"tree_{tree_count:03d}.jpg")
        cv2.imwrite(tree_filename, crop)

        print(f"🌳 Árbol guardado: {tree_filename}")

print(f"Total de árboles guardados: {tree_count}")

# --- 7. LÓGICA HÍBRIDA MEJORADA (IA + GEOMETRÍA) ---

# A. SUELO / CULTIVO (IA + Región inferior)
soil_labels = ["field-soil", "unused-land", "agriculture-land", "trees", "crop", "soil", "plant", "vegetation"]
best_soil = None
max_conf_soil = 0

for det in detections:
    if det['class'] in soil_labels:
        if det['confidence'] > max_conf_soil:
            max_conf_soil = det['confidence']
            best_soil = det

if best_soil:
    soil_result = {
        "x": int(best_soil['x']),
        "y": int(best_soil['y'])
    }
else:
    print("No se detectó suelo por IA, usando heurística de región inferior...")

    y_start = int(height * 0.6)
    y_end = height
    x_start = int(width * 0.2)
    x_end = int(width * 0.8)

    soil_result = {
        "x": int((x_start + x_end) / 2),
        "y": int((y_start + y_end) / 2)
    }

# B. CIELO (IA + Región superior)
sky_labels = ["sky", "cielo", "cloud"]
best_sky = None
max_conf_sky = 0

for det in detections:
    if det['class'] in sky_labels:
        if det['confidence'] > max_conf_sky:
            max_conf_sky = det['confidence']
            best_sky = det

if best_sky:
    sky_result = {
        "x": int(best_sky['x']),
        "y": int(best_sky['y'])
    }
else:
    print("No se detectó cielo por IA, usando heurística de región superior...")

    y_start = 0
    y_end = int(height * 0.3)
    x_start = int(width * 0.2)
    x_end = int(width * 0.8)

    sky_result = {
        "x": int((x_start + x_end) / 2),
        "y": int((y_start + y_end) / 2)
    }

# --- 8. DIBUJAR RESULTADOS ---
print("\nDibujando marcadores...")

img_vis = image_numpy.copy()

def dibujar_marcador(img, data, color, label_principal):
    x, y = data["x"], data["y"]
    texto = label_principal

    # Círculo principal
    cv2.circle(img, (x, y), 60, color, -1)
    cv2.circle(img, (x, y), 60, (255, 255, 255), 5)

    # Texto
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 3.0
    thickness = 8

    text_size, _ = cv2.getTextSize(texto, font, scale, thickness)
    text_x = x - text_size[0] // 2
    text_y = y - 80

    cv2.putText(img, texto, (text_x, text_y), font, scale, (0, 0, 0), thickness + 4)
    cv2.putText(img, texto, (text_x, text_y), font, scale, (255, 255, 255), thickness)

# Dibujar cielo (azul)
dibujar_marcador(img_vis, sky_result, (255, 0, 0), "CIELO")

# Dibujar suelo (verde)
dibujar_marcador(img_vis, soil_result, (0, 255, 0), "SUELO")

# --- 9. GUARDAR RESULTADOS ---
cv2.imwrite(OUTPUT_IMAGE, img_vis)

final_json = {
    "cielo": sky_result,
    "suelo": soil_result,
    "arboles_detectados": tree_count
}

print("\n--- JSON GENERADO ---")
print(json.dumps(final_json, indent=4))

print("\n--- IMAGEN FINAL GUARDADA ---")
print(f"Archivo: {OUTPUT_IMAGE}")
print(f"Recortes de árboles en carpeta: {TREES_DIR}")


Inicializando Roboflow...
loading Roboflow workspace...
loading Roboflow project...

Cargando imagen: prueba.insp
Intentando convertir .insp a .jpg...
✅ Imagen convertida y guardada como: convertida.jpg
✅ Imagen cargada correctamente
Shape: (2944, 5888, 3)
Min pixel: 0 Max pixel: 255
Imagen de entrada guardada como debug_entrada.jpg

Ejecutando IA...

Detecciones encontradas: 13
Clases detectadas por IA:
 - objects (0.49)
 - trees (0.42)
 - waterbodies (0.33)
 - trees (0.28)
 - field-soil (0.25)
 - unused-land (0.25)
 - objects (0.18)
 - waterbodies (0.13)
 - unused-land (0.09)
 - field-soil (0.09)
 - objects (0.09)
 - field-soil (0.07)
 - waterbodies (0.06)
🌳 Árbol guardado: arboles_detectados\tree_001.jpg
🌳 Árbol guardado: arboles_detectados\tree_002.jpg
Total de árboles guardados: 2
No se detectó cielo por IA, usando heurística de región superior...

Dibujando marcadores...

--- JSON GENERADO ---
{
    "cielo": {
        "x": 2943,
        "y": 441
    },
    "suelo": {
        "x":

In [ ]:
import cv2
import json
import numpy as np
import os
from roboflow import Roboflow
from dotenv import load_dotenv

# --- 0. CARGAR VARIABLES DE ENTORNO ---
load_dotenv()

API_KEY = os.getenv("ROBOFLOW_API_KEY")
if not API_KEY:
    # Intento de fallback si no tienes .env configurado para la prueba
    # API_KEY = "TU_API_KEY_AQUI" 
    raise ValueError("❌ No se encontró ROBOFLOW_API_KEY en el archivo .env")

# --- CONFIGURACIÓN ---
OUTPUT_IMAGE = "resultado_puntos_final.jpg"
CONVERTED_JPG = "convertida.jpg"

IMAGE_PATH = input("Introduce la ruta de la imagen (.insp o .jpg): ").strip()

# --- 1. FUNCIÓN: CONVERTIR .insp A JPG ---
def convertir_insp_a_jpg(ruta_insp, ruta_salida):
    print("Intentando convertir .insp a .jpg...")
    try:
        with open(ruta_insp, 'rb') as f:
            bytes_img = bytearray(f.read())

        numpy_array = np.asarray(bytes_img, dtype=np.uint8)
        img = cv2.imdecode(numpy_array, cv2.IMREAD_COLOR)

        if img is None:
            print("❌ No se pudo decodificar el archivo .insp.")
            return None

        cv2.imwrite(ruta_salida, img)
        print(f"✅ Imagen convertida: {ruta_salida}")
        return img
    except Exception as e:
        print("❌ Error al convertir .insp:", e)
        return None

# --- 2. CARGA DE IMAGEN ---
def cargar_imagen(ruta):
    if not os.path.exists(ruta):
        print("❌ La ruta no existe:", ruta)
        return None
    ext = os.path.splitext(ruta)[1].lower()
    if ext == ".insp":
        return convertir_insp_a_jpg(ruta, CONVERTED_JPG)
    else:
        return cv2.imread(ruta)

# --- 3. INICIALIZAR MODELO ---
print("Inicializando Roboflow...")
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("agridrone-pblcc").project("agridetect")
model = project.version(3).model

# --- 4. CARGAR DATOS ---
print(f"\nCargando imagen: {IMAGE_PATH}")
image_numpy = cargar_imagen(IMAGE_PATH)

if image_numpy is None:
    print("Error crítico: No imagen.")
    exit()

height, width, _ = image_numpy.shape
image_rgb = cv2.cvtColor(image_numpy, cv2.COLOR_BGR2RGB)

# --- 5. INFERENCIA ---
print("\nEjecutando IA...")
# Bajamos un poco la confianza mínima para intentar captar más objetos si es necesario
response = model.predict(image_rgb, confidence=5).json()

if isinstance(response, dict) and 'predictions' in response:
    detections = response['predictions']
elif isinstance(response, list):
    detections = response
else:
    detections = []

print(f"Detecciones: {len(detections)}")

# --- 6. LÓGICA DE SELECCIÓN DE PUNTOS ---

# DEFINICIÓN DE CLASES
# Ajusta estas listas según los nombres exactos de tu modelo en Roboflow
labels_sky = ["sky", "cielo", "cloud"]
labels_soil = ["field-soil", "unused-land", "soil", "dirt", "ground"] # Solo tierra
labels_crop = ["agriculture-land", "trees", "crop", "plant", "vegetation"] # Solo verde

def obtener_mejor_punto(detections, labels, heuristic_rect):
    """
    Busca la detección con mayor confianza que coincida con las labels.
    Si no encuentra, devuelve el centro del rectángulo heurístico.
    """
    best_det = None
    max_conf = 0

    # 1. Búsqueda por IA
    for det in detections:
        if det['class'] in labels:
            if det['confidence'] > max_conf:
                max_conf = det['confidence']
                best_det = det
    
    if best_det:
        return {
            "x": int(best_det['x']),
            "y": int(best_det['y'])
            "conf": best_det['confidence']
        }
    else:
        # 2. Fallback Heurístico (Centro del rectángulo dado)
        x_fallback = int((heuristic_rect['x_start'] + heuristic_rect['x_end']) / 2)
        y_fallback = int((heuristic_rect['y_start'] + heuristic_rect['y_end']) / 2)
        return {
            "x": x_fallback,
            "y": y_fallback
            "conf": 0.0
        }

# A. CIELO (Heurística: Parte superior)
sky_rect = {'x_start': int(width*0.2), 'x_end': int(width*0.8), 'y_start': 0, 'y_end': int(height*0.3)}
sky_result = obtener_mejor_punto(detections, labels_sky, sky_rect)

# B. CULTIVO (Heurística: Centro de la imagen)
# Asumimos que el cultivo suele estar en el medio o centro-bajo
crop_rect = {'x_start': int(width*0.2), 'x_end': int(width*0.8), 'y_start': int(height*0.4), 'y_end': int(height*0.7)}
crop_result = obtener_mejor_punto(detections, labels_crop, crop_rect)

# C. SUELO (Heurística: Parte inferior o bordes inferiores)
# Si hay cultivo en medio, el suelo suele verse muy abajo o en huecos
soil_rect = {'x_start': int(width*0.2), 'x_end': int(width*0.8), 'y_start': int(height*0.7), 'y_end': height}
soil_result = obtener_mejor_punto(detections, labels_soil, soil_rect)


# --- 7. DIBUJAR RESULTADOS ---
print("\nDibujando marcadores...")
img_vis = image_numpy.copy()

def dibujar_marcador(img, data, color, label_text):
    x, y = data["x"], data["y"]
    
    # Círculo
    cv2.circle(img, (x, y), 50, color, -1)     # Relleno
    cv2.circle(img, (x, y), 55, (255,255,255), 4) # Borde blanco
    
    # Texto
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 2.5
    thickness = 6
    text_size = cv2.getTextSize(texto, font, scale, thickness)[0]
    
    tx = max(0, min(x - text_size[0]//2, width - text_size[0]))
    ty = max(100, y - 70)
    
    cv2.putText(img, texto, (tx, ty), font, scale, (0,0,0), thickness+4)
    cv2.putText(img, texto, (tx, ty), font, scale, (255,255,255), thickness)

# Dibujamos en orden
dibujar_marcador(img_vis, sky_result, (255, 0, 0), "CIELO")       # Azul
dibujar_marcador(img_vis, soil_result, (0, 165, 255), "SUELO")    # Naranja
dibujar_marcador(img_vis, crop_result, (0, 255, 0), "CULTIVO")    # Verde

# --- 8. GUARDAR Y EXPORTAR ---
cv2.imwrite(OUTPUT_IMAGE, img_vis)

final_json = {
    "cielo": sky_result,
    "cultivo": crop_result,
    "suelo": soil_result,
    "dimensiones_imagen": {"w": width, "h": height}
}

print(json.dumps(final_json, indent=4))
print(f"\n✅ Imagen guardada en: {OUTPUT_IMAGE}")

Inicializando Roboflow...
loading Roboflow workspace...
loading Roboflow project...

Cargando imagen: prueba.insp
Intentando convertir .insp a .jpg...
✅ Imagen convertida: convertida.jpg

Ejecutando IA...
Detecciones: 13

Dibujando marcadores...
{
    "cielo": {
        "x": 2943,
        "y": 441,
        "metodo": "FALLBACK",
        "conf": 0.0
    },
    "cultivo": {
        "x": 3349,
        "y": 1418,
        "metodo": "IA",
        "conf": 0.41532790660858154
    },
    "suelo": {
        "x": 742,
        "y": 2237,
        "metodo": "IA",
        "conf": 0.2533997893333435
    },
    "dimensiones_imagen": {
        "w": 5888,
        "h": 2944
    }
}

✅ Imagen guardada en: resultado_puntos_final.jpg


In [1]:
import cv2
import json
import numpy as np
import os
from roboflow import Roboflow
from dotenv import load_dotenv

# --- 0. CARGAR VARIABLES DE ENTORNO ---
load_dotenv()

API_KEY = os.getenv("ROBOFLOW_API_KEY")
if not API_KEY:
    raise ValueError("❌ No se encontró ROBOFLOW_API_KEY en el archivo .env")

# --- CONFIGURACIÓN ---
OUTPUT_IMAGE = "resultado_puntos_final.jpg"
CONVERTED_JPG = "convertida.jpg"

# Pedir imagen por input
IMAGE_PATH = input("Introduce la ruta de la imagen (.insp o .jpg): ").strip()

# --- 1. FUNCIÓN: CONVERTIR .insp A JPG ---
def convertir_insp_a_jpg(ruta_insp, ruta_salida):
    print("Intentando convertir .insp a .jpg...")

    try:
        with open(ruta_insp, 'rb') as f:
            bytes_img = bytearray(f.read())

        numpy_array = np.asarray(bytes_img, dtype=np.uint8)
        img = cv2.imdecode(numpy_array, cv2.IMREAD_COLOR)

        if img is None:
            print("❌ No se pudo decodificar el archivo .insp como imagen.")
            return None

        cv2.imwrite(ruta_salida, img)
        print(f"✅ Imagen convertida y guardada como: {ruta_salida}")
        return img

    except Exception as e:
        print("❌ Error al convertir .insp:", e)
        return None

# --- 2. FUNCIÓN: CARGA ROBUSTA ---
def cargar_imagen(ruta):
    if not os.path.exists(ruta):
        print("❌ La ruta no existe:", ruta)
        return None

    ext = os.path.splitext(ruta)[1].lower()

    if ext == ".insp":
        return convertir_insp_a_jpg(ruta, CONVERTED_JPG)
    else:
        img = cv2.imread(ruta)
        if img is None:
            print("❌ OpenCV no pudo leer la imagen.")
        return img

# --- 3. INICIALIZAR MODELO ---
print("Inicializando Roboflow...")
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("agridrone-pblcc").project("agridetect")
model = project.version(3).model

# --- 4. CARGAR IMAGEN ---
print(f"\nCargando imagen: {IMAGE_PATH}")
image_numpy = cargar_imagen(IMAGE_PATH)

if image_numpy is None:
    print(json.dumps({"error": "No se pudo cargar ni convertir la imagen"}))
    exit()

print("✅ Imagen cargada correctamente")
print("Shape:", image_numpy.shape)
print("Min pixel:", image_numpy.min(), "Max pixel:", image_numpy.max())

# Guardar debug de la imagen que se usará
cv2.imwrite("debug_entrada.jpg", image_numpy)
print("Imagen de entrada guardada como debug_entrada.jpg")

# --- 5. PREPARAR PARA INFERENCIA ---
height, width, _ = image_numpy.shape
image_rgb = cv2.cvtColor(image_numpy, cv2.COLOR_BGR2RGB)

# --- 6. INFERENCIA ---
print("\nEjecutando IA...")
response = model.predict(image_rgb, confidence=5).json()

# Normalizar detecciones
if isinstance(response, dict) and 'predictions' in response:
    detections = response['predictions']
elif isinstance(response, list):
    detections = response
else:
    detections = []

print(f"Detecciones encontradas: {len(detections)}")

# --- 7. LÓGICA HÍBRIDA (IA + GEOMETRÍA) ---

# A. CULTIVO
soil_labels = ["field-soil", "unused-land", "agriculture-land", "trees", "crop", "soil"]
best_soil = None
max_conf_soil = 0

for det in detections:
    if det['class'] in soil_labels:
        if det['confidence'] > max_conf_soil:
            max_conf_soil = det['confidence']
            best_soil = det

if best_soil:
    soil_result = {
        "x": int(best_soil['x']),
        "y": int(best_soil['y']),
        "metodo": "IA (Detectado)"
    }
else:
    soil_result = {
        "x": int(width / 2),
        "y": int(height * 0.75),
        "metodo": "FALLBACK (Geometrico)"
    }

# B. CIELO
sky_labels = ["sky", "cielo", "cloud"]
best_sky = None
max_conf_sky = 0

for det in detections:
    if det['class'] in sky_labels:
        if det['confidence'] > max_conf_sky:
            max_conf_sky = det['confidence']
            best_sky = det

if best_sky:
    sky_result = {
        "x": int(best_sky['x']),
        "y": int(best_sky['y']),
        "metodo": "IA (Detectado)"
    }
else:
    sky_result = {
        "x": int(width / 2),
        "y": int(height * 0.15),
        "metodo": "FALLBACK (Geometrico)"
    }

# --- 8. DIBUJAR RESULTADOS ---
print("Dibujando marcadores...")

img_vis = image_numpy.copy()

def dibujar_marcador(img, data, color, label_principal):
    x, y = data["x"], data["y"]
    texto = f"{label_principal}: {data['metodo']}"

    cv2.circle(img, (x, y), 60, color, -1)
    cv2.circle(img, (x, y), 60, (255, 255, 255), 5)

    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 3.0
    thickness = 8

    text_size, _ = cv2.getTextSize(texto, font, scale, thickness)
    text_x = x - text_size[0] // 2
    text_y = y - 80

    cv2.putText(img, texto, (text_x, text_y), font, scale, (0, 0, 0), thickness + 4)
    cv2.putText(img, texto, (text_x, text_y), font, scale, (255, 255, 255), thickness)

# CIELO
dibujar_marcador(img_vis, sky_result, (255, 0, 0), "CIELO")

# CULTIVO
dibujar_marcador(img_vis, soil_result, (0, 255, 0), "CULTIVO")

# --- 9. GUARDAR RESULTADOS ---
cv2.imwrite(OUTPUT_IMAGE, img_vis)

final_json = {
    "cielo": sky_result,
    "cultivo": soil_result
}

print("\n--- JSON GENERADO ---")
print(json.dumps(final_json, indent=4))

print("\n--- IMAGEN FINAL GUARDADA ---")
print(f"Archivo: {OUTPUT_IMAGE}")


Inicializando Roboflow...
loading Roboflow workspace...
loading Roboflow project...

Cargando imagen: prueba.insp
Intentando convertir .insp a .jpg...
✅ Imagen convertida y guardada como: convertida.jpg
✅ Imagen cargada correctamente
Shape: (2944, 5888, 3)
Min pixel: 0 Max pixel: 255
Imagen de entrada guardada como debug_entrada.jpg

Ejecutando IA...
Detecciones encontradas: 13
Dibujando marcadores...

--- JSON GENERADO ---
{
    "cielo": {
        "x": 2944,
        "y": 441,
        "metodo": "FALLBACK (Geometrico)"
    },
    "cultivo": {
        "x": 3349,
        "y": 1418,
        "metodo": "IA (Detectado)"
    }
}

--- IMAGEN FINAL GUARDADA ---
Archivo: resultado_puntos_final.jpg
